# 03 — Transfer Learning with MobileNetV2

Two-phase training:
1. **Phase 1** — Feature extraction: MobileNetV2 base frozen, only the custom head is trained.
2. **Phase 2** — Fine-tuning: Last 30 layers of the base model unfrozen with a very low learning rate.

In [ ]:
import os, time, warnings, json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

ROOT       = r"D:\Hams Khaled\CS Year 3 Sem 2\Image Processing\Final Project\project"
SPLIT_DIR  = os.path.join(ROOT, "data",    "split")
MODELS_DIR = os.path.join(ROOT, "models")
PLOTS_DIR  = os.path.join(ROOT, "results", "plots")
CKPT_PATH  = os.path.join(MODELS_DIR, "transfer_model_phase1.h5")
MODEL_PATH = os.path.join(MODELS_DIR, "transfer_model.h5")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR,  exist_ok=True)

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
print(f"  TensorFlow version: {tf.__version__}")
print("  ✔ Configuration complete.")


## Step 1 — Data Generators

Same as notebook 02 — self-contained.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255, horizontal_flip=True,
    rotation_range=20, zoom_range=0.2, brightness_range=[0.8, 1.2]
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, "train"),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", shuffle=True, seed=RANDOM_STATE
)
val_gen = val_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, "val"),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", shuffle=False
)
print(f"  Train samples: {train_gen.samples}")
print(f"  Val   samples: {val_gen.samples}")


## Phase 1 — Feature Extraction

The MobileNetV2 base is frozen (`trainable=False`). Only the custom head layers get updated.

In [ ]:
# Load MobileNetV2 pre-trained on ImageNet
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False   # freeze entire base

print(f"  Base model layers   : {len(base_model.layers)}")
print(f"  Base model trainable: {base_model.trainable}")

# Build custom head
x      = base_model.output
x      = Flatten()(x)
x      = Dense(256, activation='relu')(x)
x      = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model  = Model(inputs=base_model.input, outputs=output, name="FreshRottenCNN_Transfer")

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

trainable_p1 = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f"  Trainable params (Phase 1): {trainable_p1:,}")


In [ ]:
cb_phase1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(filepath=CKPT_PATH, monitor='val_loss', save_best_only=True, verbose=1),
]

print("  ▶ Phase 1: Training custom head (15 epochs) …")
t_start = time.time()

history_p1 = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=cb_phase1,
    verbose=1,
)

t_phase1 = time.time() - t_start
print(f"  ✔ Phase 1 complete in {t_phase1:.1f} s")


## Phase 2 — Fine-tuning

Unfreeze the last 30 layers of the base model and retrain with a much smaller learning rate (`1e-5`) to avoid destroying ImageNet features.

In [ ]:
# Unfreeze last 30 layers of the backbone
for layer in base_model.layers[-30:]:
    layer.trainable = True

trainable_p2 = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f"  Trainable params (Phase 2): {trainable_p2:,}")
print(f"  Unfrozen layers: {sum(1 for l in base_model.layers if l.trainable)} / {len(base_model.layers)}")

# Recompile with low lr
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

cb_phase2 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(filepath=MODEL_PATH, monitor='val_loss', save_best_only=True, verbose=1),
]

print("  ▶ Phase 2: Fine-tuning last 30 layers (15 epochs) …")
t2_start = time.time()

history_p2 = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=cb_phase2,
    verbose=1,
)

t_phase2 = time.time() - t2_start
t_total  = t_phase1 + t_phase2
print(f"  ✔ Phase 2 complete in {t_phase2:.1f} s")
print(f"  ✔ Total training time: {t_total:.1f} s ({t_total/60:.1f} min)")
print(f"  ✔ Final model saved → {MODEL_PATH}")


## Step 3 — Plot Training Curves

We concatenate Phase 1 and Phase 2 history for a single view, and also save them separately.

In [ ]:
def save_curve(vals_train, vals_val, metric, out_path, title):
    epochs_ran = range(1, len(vals_train) + 1)
    plt.figure(figsize=(10, 5))
    plt.plot(epochs_ran, vals_train, 'b-o', label=f'Train {metric.capitalize()}')
    plt.plot(epochs_ran, vals_val,   'r-o', label=f'Val {metric.capitalize()}')
    n_p1 = len(history_p1.history[metric])
    plt.axvline(x=n_p1, color='gray', linestyle='--', label='Phase 1 → Phase 2')
    plt.title(title); plt.xlabel('Epoch'); plt.ylabel(metric.capitalize())
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(out_path, dpi=150); plt.close()
    print(f"  ✔ Saved → {out_path}")

# Concatenate both phases
acc_train = history_p1.history['accuracy']  + history_p2.history['accuracy']
acc_val   = history_p1.history['val_accuracy'] + history_p2.history['val_accuracy']
loss_train= history_p1.history['loss']      + history_p2.history['loss']
loss_val  = history_p1.history['val_loss']  + history_p2.history['val_loss']

save_curve(acc_train,  acc_val,  "accuracy",
           os.path.join(PLOTS_DIR, "transfer_accuracy.png"),
           "Transfer Learning — Training vs Validation Accuracy (Phase 1 + 2)")

save_curve(loss_train, loss_val, "loss",
           os.path.join(PLOTS_DIR, "transfer_loss.png"),
           "Transfer Learning — Training vs Validation Loss (Phase 1 + 2)")


## Step 4 — Summary

In [ ]:
epochs_ran_total = (len(history_p1.history['accuracy']) +
                    len(history_p2.history['accuracy']))

print("=" * 55)
print("  TRANSFER LEARNING MODEL SUMMARY")
print("=" * 55)
print(f"  Phase 1 epochs    : {len(history_p1.history['accuracy'])}")
print(f"  Phase 2 epochs    : {len(history_p2.history['accuracy'])}")
print(f"  Total epochs      : {epochs_ran_total}")
print(f"  Trainable (Ph 1)  : {trainable_p1:,}")
print(f"  Trainable (Ph 2)  : {trainable_p2:,}")
print(f"  Total train time  : {t_total:.1f} s ({t_total/60:.1f} min)")
print(f"  Best val accuracy : {max(acc_val):.4f}")
print("  ✔ Model saved →", MODEL_PATH)

meta = {
    "transfer_training_time": t_total,
    "transfer_epochs":        epochs_ran_total,
    "transfer_params_p1":     int(trainable_p1),
    "transfer_params_p2":     int(trainable_p2),
}
with open(os.path.join(ROOT, "results", "transfer_meta.json"), "w") as f:
    json.dump(meta, f)
print("  ✔ Metadata saved.")
